# Descarga del Dataset
## Generador de Reportes de Partidos de Fútbol — Premier League

Este notebook descarga automáticamente `Matches.csv` desde Kaggle y lo guarda en `data/raw/`,
la misma ubicación indicada en el README del proyecto.

**Dataset:** [Club Football Match Data (2000–2025)](https://www.kaggle.com/datasets/adamgbor/club-football-match-data-2000-2025)

Ejecutar este notebook **antes** de `01_eda.ipynb` si aún no tenés el archivo localmente.

## Requisitos previos

1. Crear una cuenta en [Kaggle](https://www.kaggle.com/).
2. Ir a **Account → API → Create New Token**. Se descargará `kaggle.json`.
3. Configurar las credenciales:

   **Mediante variables de entorno en `.env`** (en la raíz del proyecto)
   ```
   KAGGLE_USERNAME=tu_usuario
   KAGGLE_KEY=tu_api_key
   ```

4. Aceptar las reglas del dataset en Kaggle (botón **"Download"** o **"New Notebook"** en la página del dataset).

In [ ]:
from pathlib import Path

from dotenv import load_dotenv
from kaggle.api.kaggle_api_extended import KaggleApi

PROJECT_ROOT = Path("..").resolve()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_FILE = RAW_DIR / "Matches.csv"
DATASET = "adamgbor/club-football-match-data-2000-2025"
FILENAME = "Matches.csv"

load_dotenv(PROJECT_ROOT / ".env")
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"Destino: {OUTPUT_FILE}")

In [ ]:
import zipfile

if OUTPUT_FILE.exists():
    size_mb = OUTPUT_FILE.stat().st_size / (1024 * 1024)
    print(f"El archivo ya existe ({size_mb:.1f} MB).")
    print("Si querés volver a descargarlo, borrá el archivo y ejecutá esta celda de nuevo.")
else:
    api = KaggleApi()
    api.authenticate()

    print(f"Descargando {FILENAME} desde Kaggle...")
    api.dataset_download_file(
        DATASET,
        FILENAME,
        path=str(RAW_DIR),
        force=True,
    )

    zip_path = RAW_DIR / f"{FILENAME}.zip"
    if not OUTPUT_FILE.exists() and zip_path.exists():
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall(RAW_DIR)
        zip_path.unlink()

    if not OUTPUT_FILE.exists():
        raise FileNotFoundError(
            f"No se encontró {OUTPUT_FILE} después de la descarga. "
            "Verificá tus credenciales de Kaggle y que aceptaste las reglas del dataset."
        )

    size_mb = OUTPUT_FILE.stat().st_size / (1024 * 1024)
    print(f"Descarga completada: {OUTPUT_FILE} ({size_mb:.1f} MB)")

In [ ]:
import pandas as pd

df = pd.read_csv(OUTPUT_FILE, low_memory=False)
epl = df[df["Division"] == "E0"]

print(f"Filas totales: {len(df):,}")
print(f"Partidos Premier League (E0): {len(epl):,}")
print(f"Columnas: {df.shape[1]}")
print("\nListo. Podés continuar con notebooks/01_eda.ipynb.")